# 07 - Model-Version Tagging and Mixed-Version Batch Detection Demo

Companion notebook to `../07-production-resilience-and-operational-engineering.md`, specifically bug
narrative #3 ("a retrained model deployed without any version identifier changing anywhere") and the
chapter's hardening-gap fix (read `MODEL_VERSION` from an environment variable and stamp it on every
response, replacing the misleading `model_loaded_at` cold-start timestamp). It also implements the
concrete mechanism behind `06-detection-deduplication-and-model-version-drift.md`'s `model_version`
field from the proposed `detection_runs` design.

Extends `04_lambda_inference_handler_demo.ipynb`'s `StubChartDetectorModel` / `lambda_handler` pattern.
Fully offline -- only `PIL`, `numpy`, and the standard library.

In [1]:
import io
import time

from PIL import Image
import numpy as np

print("Ready.")

Ready.


## 1. The bug, reproduced: `model_loaded_at` looks like a version signal but isn't one

Chapter 07 bug #3, and the hardening gap, state this plainly: `model_loaded_at` is a cold-start
timestamp -- it changes on **every** fresh Lambda execution environment, regardless of whether the
underlying `weights.pt` artifact changed at all. The cell below "cold-starts" two separate execution
environments that both load the exact same model artifact, and shows their `model_loaded_at` values
differ anyway -- proving the field cannot be used, even informally, to tell "same model" from
"different model." 

In [2]:
class StubChartDetectorModelUntagged:
    """The pre-fix handler shape (chapter 07, before the hardening fix): the only version-adjacent
    field is model_loaded_at, a cold-start wall-clock timestamp with no relationship to the actual
    weights.pt content."""

    def __init__(self):
        self.loaded_at = time.time()

    def predict(self, image: Image.Image, confidence_threshold: float = 0.25):
        return [{"class": "chart", "confidence": 0.96, "bbox_pixels": [40, 45, 180, 165]}]


# Two separate Lambda execution environments cold-start, both loading the SAME underlying
# weights.pt artifact -- no retrain happened between them.
env_1_model = StubChartDetectorModelUntagged()
time.sleep(0.01)
env_2_model = StubChartDetectorModelUntagged()

print("env_1 model_loaded_at:", env_1_model.loaded_at)
print("env_2 model_loaded_at:", env_2_model.loaded_at)
assert env_1_model.loaded_at != env_2_model.loaded_at, \
    "cold-start timestamps differ even though both loaded the identical model artifact"
print("\nConfirmed: model_loaded_at differs across environments even with ZERO underlying model")
print("change -- it cannot be used to detect or attribute a real retrain.")

env_1 model_loaded_at: 1784787047.3285604
env_2 model_loaded_at: 1784787047.3388627

Confirmed: model_loaded_at differs across environments even with ZERO underlying model
change -- it cannot be used to detect or attribute a real retrain.


## 2. The fix: `MODEL_VERSION` read from environment/config at cold start, stamped on every result

Chapter 07's hardening-gap fix, and chapter 06 Part 4's `model_version` field: read a stable model
identity (an explicit tag tied to the artifact, not a timestamp) from environment configuration once at
cold start, and stamp it on every response -- so two environments loading the same artifact report the
*same* identity, and a real retrain is the only thing that changes it.

In [3]:
class StubChartDetectorModelTagged:
    """The fixed handler shape: MODEL_VERSION is read from environment/config at cold-start init --
    an explicit tag tied to the deployed artifact, e.g. set by the deployment pipeline whenever the
    model's hash changes (chapter 07's proposed CI gate) -- and stamped on every prediction result."""

    def __init__(self, model_version: str):
        self.model_version = model_version   # NOT a timestamp -- an explicit, deployment-set tag
        self.loaded_at = time.time()          # kept for operational/debugging purposes only, never
                                               # used for version identity

    def predict(self, image: Image.Image, confidence_threshold: float = 0.25):
        return [{
            "class": "chart",
            "confidence": 0.96,
            "bbox_pixels": [40, 45, 180, 165],
            "model_version": self.model_version,   # stamped on every individual detection result
        }]


# Same scenario as section 1: two environments cold-start, both configured with the SAME
# MODEL_VERSION because no retrain has happened.
env_1_tagged = StubChartDetectorModelTagged(model_version="yolov5-chart-finetune-2026-03-01")
time.sleep(0.01)
env_2_tagged = StubChartDetectorModelTagged(model_version="yolov5-chart-finetune-2026-03-01")

print("env_1 model_version:", env_1_tagged.model_version, "| loaded_at:", env_1_tagged.loaded_at)
print("env_2 model_version:", env_2_tagged.model_version, "| loaded_at:", env_2_tagged.loaded_at)
assert env_1_tagged.model_version == env_2_tagged.model_version
assert env_1_tagged.loaded_at != env_2_tagged.loaded_at  # timestamps still differ -- that's expected

print("\nConfirmed: model_version agrees across environments despite different cold-start times --")
print("this is the signal that actually reflects whether the deployed artifact changed.")

env_1 model_version: yolov5-chart-finetune-2026-03-01 | loaded_at: 1784787047.3454943
env_2 model_version: yolov5-chart-finetune-2026-03-01 | loaded_at: 1784787047.3560896

Confirmed: model_version agrees across environments despite different cold-start times --
this is the signal that actually reflects whether the deployed artifact changed.


## 3. A retrain happens mid-fleet, exactly like course 09's Lambda-cache caveat

A new fine-tune (improved recall on forest plots) is promoted. Some execution environments are already
warm and were cold-started before the promotion; a fresh one cold-starts after. Every response, from
every environment, carries its own actual `model_version` -- making the mixed-version window during the
rollout directly attributable per result, the same shape of fix as `07-production-resilience...` in
course 09's Claim Extraction pipeline, applied here to detections instead of classifications.

In [4]:
synthetic_image = Image.fromarray(
    (np.random.default_rng(0).random((300, 400, 3)) * 255).astype(np.uint8)
)

# Environments already warm from before the retrain promotion.
env_a = StubChartDetectorModelTagged(model_version="yolov5-chart-finetune-2026-03-01")
env_b = StubChartDetectorModelTagged(model_version="yolov5-chart-finetune-2026-03-01")

# The retraining pipeline promotes a new fine-tune -- a fresh environment picks it up on cold start.
env_c = StubChartDetectorModelTagged(model_version="yolov5-chart-finetune-2026-06-14")

def run_detection(model, image, document_id, client):
    detections = model.predict(image)
    return {"document_id": document_id, "client": client, "detections": detections,
            "model_version": model.model_version}


batch = [
    run_detection(env_a, synthetic_image, "doc-101", "eli-lilly"),
    run_detection(env_c, synthetic_image, "doc-102", "eli-lilly"),
    run_detection(env_b, synthetic_image, "doc-103", "eli-lilly"),
    run_detection(env_c, synthetic_image, "doc-104", "eli-lilly"),
    run_detection(env_c, synthetic_image, "doc-105", "eli-lilly"),
]

for r in batch:
    print(f"{r['document_id']}  model_version={r['model_version']}")

doc-101  model_version=yolov5-chart-finetune-2026-03-01
doc-102  model_version=yolov5-chart-finetune-2026-06-14
doc-103  model_version=yolov5-chart-finetune-2026-03-01
doc-104  model_version=yolov5-chart-finetune-2026-06-14
doc-105  model_version=yolov5-chart-finetune-2026-06-14


## 4. Flagging a mixed-version batch -- the downstream-reporting tie-in

This is `06-detection-deduplication-and-model-version-drift.md` Part 5's concrete concern: a
monthly/report-window detection count that doesn't segment by `model_version` can't tell "the client
sent more documents" apart from "the model changed mid-window." The function below is the reporting-side
consumer of the tag: it flags any reporting window containing more than one distinct `model_version`,
and reports counts broken out per version so the two explanations are never conflated.

In [5]:
def analyze_batch_for_mixed_versions(results: list[dict]) -> dict:
    """The reporting-layer check chapter 06 Part 5 says has to exist downstream of the tag: segment
    counts by model_version and flag whenever a single window spans more than one."""
    versions_seen = {}
    for r in results:
        versions_seen.setdefault(r["model_version"], []).append(r["document_id"])

    return {
        "distinct_model_versions": len(versions_seen),
        "counts_by_version": {v: len(docs) for v, docs in versions_seen.items()},
        "documents_by_version": versions_seen,
        "mixed_version_window": len(versions_seen) > 1,
    }


report = analyze_batch_for_mixed_versions(batch)
print("Distinct model versions in this window:", report["distinct_model_versions"])
print("Counts by version:", report["counts_by_version"])
print("Mixed-version window flagged:", report["mixed_version_window"])

assert report["mixed_version_window"] is True
assert report["counts_by_version"] == {
    "yolov5-chart-finetune-2026-03-01": 2,
    "yolov5-chart-finetune-2026-06-14": 3,
}
print("\nConfirmed: a detection-count spike in this window is now explainable -- 3 of 5 documents ran")
print("against the newly promoted model, 2 against the outgoing one -- rather than an unattributable")
print("jump a stakeholder would otherwise have to guess about.")

Distinct model versions in this window: 2
Counts by version: {'yolov5-chart-finetune-2026-03-01': 2, 'yolov5-chart-finetune-2026-06-14': 3}
Mixed-version window flagged: True

Confirmed: a detection-count spike in this window is now explainable -- 3 of 5 documents ran
against the newly promoted model, 2 against the outgoing one -- rather than an unattributable
jump a stakeholder would otherwise have to guess about.


## 5. Contrast: the same batch, without tagging -- the question becomes unanswerable

To make the value of the tag concrete rather than assumed, the cell below reruns an equivalent batch
through the untagged (`model_loaded_at`-only) handler from Section 1, and shows the mixed-version
question simply cannot be answered from the results -- even though the same underlying mixed-version
rollout happened.

In [6]:
def run_detection_untagged(model, image, document_id, client):
    detections = model.predict(image)
    return {"document_id": document_id, "client": client, "detections": detections,
            "model_loaded_at": model.loaded_at}


untagged_env_a = StubChartDetectorModelUntagged()
untagged_env_c = StubChartDetectorModelUntagged()  # stands in for the post-retrain cold start

untagged_batch = [
    run_detection_untagged(untagged_env_a, synthetic_image, "doc-101", "eli-lilly"),
    run_detection_untagged(untagged_env_c, synthetic_image, "doc-102", "eli-lilly"),
    run_detection_untagged(untagged_env_a, synthetic_image, "doc-103", "eli-lilly"),
]

for r in untagged_batch:
    print(f"{r['document_id']}  model_loaded_at={r['model_loaded_at']}")

# Every model_loaded_at value is different -- but that's just cold-start jitter, not evidence of a
# real model change, and there is no field at all naming which artifact actually ran.
distinct_timestamps = {r["model_loaded_at"] for r in untagged_batch}
can_attribute_to_a_real_model_change = "model_version" in untagged_batch[0]

print("\nDistinct model_loaded_at values:", len(distinct_timestamps), "(all different, as always)")
print("Can this batch answer 'did the model actually change'?", can_attribute_to_a_real_model_change)
assert not can_attribute_to_a_real_model_change
print("\nConfirmed: without an explicit model_version tag, every cold start looks like a 'change' by")
print("model_loaded_at's own logic, and there is no way to tell a real retrain apart from ordinary")
print("container recycling -- exactly the false signal chapter 07's bug #3 describes.")

doc-101  model_loaded_at=1784787047.3845763
doc-102  model_loaded_at=1784787047.3845875
doc-103  model_loaded_at=1784787047.3845763

Distinct model_loaded_at values: 2 (all different, as always)
Can this batch answer 'did the model actually change'? False

Confirmed: without an explicit model_version tag, every cold start looks like a 'change' by
model_loaded_at's own logic, and there is no way to tell a real retrain apart from ordinary
container recycling -- exactly the false signal chapter 07's bug #3 describes.


## Takeaways

- **A cold-start timestamp is not a version signal, even informally.** Section 1 proves two identical,
  never-retrained environments still report different `model_loaded_at` values -- the field varies for
  a reason completely unrelated to whether the model changed.
- **`MODEL_VERSION` read from environment/config at cold start is stable across environments loading the
  same artifact, and only changes when the artifact actually does** (Section 2) -- the property that
  makes it usable for attribution at all.
- **Tagging every individual result, not just logging a deploy event, is what makes a mixed-version
  rollout window attributable per-document**, not just detectable in aggregate after the fact (Sections
  3-4).
- **The reporting layer has to actively consume the tag** (`analyze_batch_for_mixed_versions`) for any
  of this to matter to a stakeholder reading a monthly count -- Section 5 shows the exact same rollout
  happening with the tag absent, and the question "did the model change" becomes unanswerable from the
  data alone, matching chapter 06 Part 5's downstream-reporting concern precisely.